In [18]:
import numpy as np
import pandas as pd
import os, sys, shutil, subprocess, zipfile, random
from pathlib import Path
from datetime import datetime

In [19]:
COMP = "playground-series-s6e11"
SEED = 42
USE_DRIVE_CACHE = False

In [20]:
def detect_env():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    try:
        import google.colab
        return "colab"
    except ImportError:
        return "local"

ENV = detect_env()

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"Environment: {ENV}")

Environment: local


In [21]:
def find_repo_root(start: Path = Path.cwd()) -> Path:
    """Walk up from the notebook folder until we find the git repo root."""
    for p in [start, *start.parents]:
        if (p / ".git").exists():
            return p
    return start

if ENV == "kaggle":
    DATA_DIR = None
    OUT_DIR = Path("/kaggle/working")
elif ENV == "colab":
    if USE_DRIVE_CACHE:
        from google.colab import drive
        drive.mount("/content/drive")
        DATA_DIR = Path(f"/content/drive/MyDrive/kaggle-data/{COMP}")
    else:
        DATA_DIR = Path(f"/content/data/{COMP}")
    OUT_DIR = Path("/content/output")
else:
    ROOT = find_repo_root()
    DATA_DIR = ROOT / "data" / COMP
    OUT_DIR = ROOT / "submissions"

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output dir: {OUT_DIR}")

Output dir: /home/the_r/kaggle/kaggle-playground-s6e11-fraud/submissions


In [22]:
def setup_credentials():
    if ENV == "kaggle":
        return  # not needed on Kaggle

    if ENV == "colab":
        from google.colab import userdata
        try:
            os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")      # new-style token
        except Exception:
            os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")        # legacy kaggle.json values
            os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
        return

    # local: accept any supported credential
    kaggle_dir = Path.home() / ".kaggle"
    found = {
        "KAGGLE_API_TOKEN env var": bool(os.environ.get("KAGGLE_API_TOKEN")),
        "~/.kaggle/access_token": (kaggle_dir / "access_token").exists()
                                  or (kaggle_dir / "access_token.txt").exists(),
        "legacy env vars": bool(os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")),
        "legacy ~/.kaggle/kaggle.json": (kaggle_dir / "kaggle.json").exists(),
    }
    used = [k for k, v in found.items() if v]
    if not used:
        raise RuntimeError(
            "No Kaggle credentials found. Generate a token at kaggle.com → Settings → API "
            "and save it to ~/.kaggle/access_token, or run: kaggle auth login"
        )
    print(f"Kaggle credentials found: {', '.join(used)}")

setup_credentials()

Kaggle credentials found: ~/.kaggle/access_token


In [23]:
def find_kaggle_input(comp: str) -> Path:
    """Locate attached competition data on Kaggle."""
    base = Path("/kaggle/input")
    for candidate in (base / comp, base / "competitions" / comp):
        if candidate.exists():
            return candidate
    # fallback: first folder containing data files
    for p in base.rglob("*"):
        if p.suffix in {".csv", ".parquet"}:
            return p.parent
    raise FileNotFoundError("Competition data not attached. Use '+ Add Input' in the notebook sidebar.")

def kaggle_cli() -> str:
    """Find the kaggle CLI, including inside a venv that isn't on PATH."""
    exe = shutil.which("kaggle")
    if exe:
        return exe
    venv_exe = Path(sys.executable).parent / ("kaggle.exe" if os.name == "nt" else "kaggle")
    if venv_exe.exists():
        return str(venv_exe)
    raise RuntimeError("kaggle CLI not found. Install it: uv pip install kaggle")

def download_data(comp: str, dest: Path):
    dest.mkdir(parents=True, exist_ok=True)
    if any(dest.glob("*.csv")) or any(dest.glob("*.parquet")):
        print(f"Data already in {dest}, skipping download.")
        return

    print(f"Downloading {comp} ...")
    result = subprocess.run(
        [kaggle_cli(), "competitions", "download", "-c", comp, "-p", str(dest)],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(
            "Download failed. Check you've joined the competition and accepted its rules "
            "on kaggle.com, and that your credentials are correct."
        )

    for z in dest.glob("*.zip"):
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        z.unlink()
    print("Done.")

if ENV == "kaggle":
    DATA_DIR = find_kaggle_input(COMP)
else:
    download_data(COMP, DATA_DIR)

print(f"Data dir: {DATA_DIR}")
for p in sorted(DATA_DIR.iterdir()):
    print(f"  {p.name:40s} {p.stat().st_size / 1e6:8.1f} MB")

Data already in /home/the_r/kaggle/kaggle-playground-s6e11-fraud/data/playground-series-s6e11, skipping download.
Data dir: /home/the_r/kaggle/kaggle-playground-s6e11-fraud/data/playground-series-s6e11
  sample_submission.csv                         8.4 MB
  test.csv                                     41.1 MB
  train.csv                                    98.5 MB


In [24]:
def load(name: str):
    """Load name.csv or name.parquet if it exists."""
    for ext in (".csv", ".parquet"):
        p = DATA_DIR / f"{name}{ext}"
        if p.exists():
            return pd.read_csv(p) if ext == ".csv" else pd.read_parquet(p)
    print(f"(no {name} file found)")
    return None

train = load("train")
test = load("test")
sample_sub = load("sample_submission")

for name, df in [("train", train), ("test", test), ("sample_submission", sample_sub)]:
    if df is not None:
        print(f"{name:18s} {df.shape}")
train.head() if train is not None else None

train              (699960, 26)
test               (299984, 25)
sample_submission  (299984, 2)


,id,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,...,auth_method,channel,device_type,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt,is_fraud
0,0,61.16,9.77,1,17.78,48,56,1269.35,0,5.5,...,OTP,POS,iPhone,False,False,False,False,False,False,0.0
1,1,24.83,1.54,3,13.06,38,28,3233.00,0,33.4,...,PIN,In-App,Android Phone,False,False,False,False,False,False,0.0
2,2,6.36,1.20,4,8.17,50,55,1046.88,0,35.8,...,OTP,Online,Tablet,False,False,True,False,False,False,0.0
3,3,37.44,5.13,4,8.67,48,48,2217.67,0,20.7,...,3D Secure,POS,Android Phone,False,False,False,False,False,False,0.0
4,4,653.85,0.80,4,11.69,37,52,3291.06,0,34.8,...,3D Secure,Online,Windows PC,False,False,False,False,False,False,0.0


In [26]:
def save_submission(df: pd.DataFrame, tag: str = ""):
    """Always writes submission.csv. Locally, also keeps a timestamped copy."""
    if sample_sub is not None:
        assert list(df.columns) == list(sample_sub.columns), "Columns don't match sample_submission"
        assert len(df) == len(sample_sub), "Row count doesn't match sample_submission"

    path = OUT_DIR / "submission.csv"
    df.to_csv(path, index=False)
    print(f"Saved {path}")

    if ENV == "local":
        stamp = datetime.now().strftime("%Y%m%d_%H%M")
        archived = OUT_DIR / f"sub_{stamp}{'_' + tag if tag else ''}.csv"
        df.to_csv(archived, index=False)
        print(f"Archived {archived}")

# Example once you have predictions:
sub = sample_sub.copy()
# sub[sub.columns[1]] = preds
# save_submission(sub, tag="lgbm_baseline")

In [27]:
sub

,id,is_fraud
0,699960,0.0167
1,699961,0.0167
2,699962,0.0167
3,699963,0.0167
4,699964,0.0167
...,...,...
299979,999939,0.0167
299980,999940,0.0167
299981,999941,0.0167
299982,999942,0.0167
